# Phase 4: Check the client's claims

This notebook checks the numerical statements and observations in `BRIEF.md` against the raw bookings. It does not clean rows, classify columns, train a model, or choose production features.

The vendor claims cannot be verified because the supplied data has no vendor predictions or front-desk usage records.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

SRC_DIR = Path.cwd().parent / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
import data

bookings = data.read_raw()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
print(f"Rows: {len(bookings):,}; source columns: {bookings.shape[1] - 2}")
print(f"Arrival dates: {bookings['arrival_date'].min().date()} to {bookings['arrival_date'].max().date()}")

overall = pd.DataFrame({"bookings": [len(bookings)], "cancelled": [int(bookings["is_canceled"].sum())], "cancellation_rate": [bookings["is_canceled"].mean()]}, index=["All hotels"])
by_hotel = bookings.groupby("hotel")["is_canceled"].agg(bookings="size", cancelled="sum", cancellation_rate="mean")
print("Claim 1: about one booking in four is cancelled")
display(overall)
display(by_hotel)

by_deposit = bookings.groupby("deposit_type")["is_canceled"].agg(bookings="size", cancelled="sum", cancellation_rate="mean")
by_hotel_deposit = bookings.groupby(["hotel", "deposit_type"])["is_canceled"].agg(bookings="size", cancelled="sum", cancellation_rate="mean")
print("Claim 2: non-refundable bookings are guaranteed")
display(by_deposit)
display(by_hotel_deposit)

segment_rates = bookings.groupby("market_segment")["is_canceled"].agg(bookings="size", cancelled="sum", cancellation_rate="mean").sort_values("cancellation_rate", ascending=False)
customer_rates = bookings.groupby("customer_type")["is_canceled"].agg(bookings="size", cancelled="sum", cancellation_rate="mean").sort_values("cancellation_rate", ascending=False)
print("Descriptive market-segment rates for the wishlist; no hidden-type model is built")
display(segment_rates)
print("Descriptive customer-type rates for the wishlist; no hidden-type model is built")
display(customer_rates)

periods = pd.Series(pd.NA, index=bookings.index, dtype="string")
periods.loc[bookings["arrival_date"] < "2016-07-01"] = "train"
periods.loc[(bookings["arrival_date"] >= "2016-07-01") & (bookings["arrival_date"] < "2017-01-01")] = "choose"
periods.loc[bookings["arrival_date"] >= "2017-01-01"] = "proof"
period_summary = bookings.assign(period=periods).groupby("period", observed=False)["is_canceled"].agg(bookings="size", cancelled="sum", cancellation_rate="mean")
print("Chronological period counts")
display(period_summary)

print("Policy inputs: walk cost = EUR 300; ceiling = 10 walks per 1,000. These come from the contract, not the booking rows.")


Rows: 119,390; source columns: 32
Arrival dates: 2015-07-01 to 2017-08-31
Claim 1: about one booking in four is cancelled


,bookings,cancelled,cancellation_rate
All hotels,119390,44224,0.370416


,bookings,cancelled,cancellation_rate
hotel,,,
City Hotel,79330,33102,0.417270
Resort Hotel,40060,11122,0.277634


Claim 2: non-refundable bookings are guaranteed


,bookings,cancelled,cancellation_rate
deposit_type,,,
No Deposit,104641,29694,0.283770
Non Refund,14587,14494,0.993624
Refundable,162,36,0.222222


bookings  cancelled  cancellation_rate
hotel        deposit_type                                        
City Hotel   No Deposit       66442      20244           0.304687
             Non Refund       12868      12844           0.998135
             Refundable          20         14           0.700000
Resort Hotel No Deposit       38199       9450           0.247389
             Non Refund        1719       1650           0.959860
             Refundable         142         22           0.154930

... (3 outputs left out)

,bookings,cancelled,cancellation_rate
customer_type,,,
Transient,89613,36514,0.407463
Contract,4076,1262,0.309617
Transient-Party,25124,6389,0.254299
Group,577,59,0.102253


Chronological period counts


,bookings,cancelled,cancellation_rate
period,,,
choose,29546,10894,0.368713
proof,40687,15745,0.386979
train,49157,17585,0.357731


Policy inputs: walk cost = EUR 300; ceiling = 10 walks per 1,000. These come from the contract, not the booking rows.


## Conclusions

- The observed overall cancellation rate is 37.04%, not 25%; hotel rates differ.
- `Non Refund` has a 99.36% cancellation rate in this dataset, so the guarantee observation is contradicted.
- The two vendor claims cannot be verified from the supplied data.
- Segment tables are descriptive only; hidden guest-type discovery is deferred.
- Findings are recorded in `docs/CHARTER.md`, `docs/DECISIONS.md`, and `docs/PHASE4_FINDINGS.md`.